In [23]:
import pandas as pd
import numpy as np
import faiss
import networkx as nx

from utils.embeddings import Embeddings
from utils.load import LoadDataCSV

# Load datasets

df_loader=LoadDataCSV()

## Graph Construction

In [32]:
def compute_similarity(embedding_a, embedding_b, method='cosine'):
    """
    Compute similarity between two sets of embeddings using the specified method.
    Args:
        embedding_a (np.ndarray): A 2D numpy array for set A embeddings.
        embedding_b (np.ndarray): A 2D numpy array for set B embeddings.
        method (str): Similarity method ('cosine', 'l2', 'faiss').
    Returns:
        np.ndarray: Similarity or distance matrix.
    """
    if method == 'cosine':
        # Cosine similarity
        norm_a = np.linalg.norm(embedding_a, axis=1, keepdims=True)
        norm_b = np.linalg.norm(embedding_b, axis=1, keepdims=True)
        return np.dot(embedding_a, embedding_b.T) / (norm_a * norm_b.T)
    elif method == 'l2':
        # Euclidean distance
        diff = embedding_a[:, np.newaxis, :] - embedding_b[np.newaxis, :, :]
        return -np.linalg.norm(diff, axis=2)  # Negative to treat smaller as better
    elif method == 'faiss':
        # Faiss IndexFlatL2
        index = faiss.IndexFlatL2(embedding_b.shape[1])
        index.add(embedding_b)
        _, distances = index.search(embedding_a, embedding_b.shape[0])
        return -distances  # Negative for similarity-like behavior
    else:
        raise ValueError("Unsupported similarity method. Use 'cosine', 'l2', or 'faiss'.")


def build_graph(posts_df, fact_checks_df, method='cosine', threshold=0.7, post_embedding_col='chunk_embeddings', fact_embedding_col='embedding'):
    """
    Builds a graph where nodes represent posts and fact-checks, and edges represent similarity between them.
    Args:
        posts_df (pd.DataFrame): DataFrame containing posts with embeddings.
        fact_checks_df (pd.DataFrame): DataFrame containing fact-checks with embeddings.
        method (str): Similarity computation method ('cosine', 'l2', 'faiss').
        threshold (float): Similarity threshold for creating edges.
        post_embedding_col (str): Column name in posts_df containing post embeddings.
        fact_embedding_col (str): Column name in fact_checks_df containing fact embeddings.
    Returns:
        networkx.Graph: A graph with posts and fact-checks as nodes and similarity-based edges.
    """
    G = nx.Graph()

    # Add nodes for posts and fact-checks
    G.add_nodes_from(posts_df['post_id'], node_type='post')
    G.add_nodes_from(fact_checks_df['fact_check_id'], node_type='fact_check')

    # Compute post-to-post similarity
    posts_embeddings = np.vstack(posts_df[post_embedding_col].apply(np.mean, axis=0))
    similarity_matrix = compute_similarity(posts_embeddings, posts_embeddings, method)

    # Add edges for highly similar posts
    for i, post_id_a in enumerate(posts_df['post_id']):
        for j, post_id_b in enumerate(posts_df['post_id']):
            if i != j and similarity_matrix[i, j] > threshold:
                G.add_edge(post_id_a, post_id_b, weight=similarity_matrix[i, j])

    # Compute post-to-fact-check similarity
    fact_embeddings = np.vstack(fact_checks_df[fact_embedding_col])
    similarity_matrix_post_fact = compute_similarity(posts_embeddings, fact_embeddings, method)

    # Add edges between posts and relevant fact-checks
    for i, post_id in enumerate(posts_df['post_id']):
        for j, fact_check_id in enumerate(fact_checks_df['fact_check_id']):
            if similarity_matrix_post_fact[i, j] > threshold:
                G.add_edge(post_id, fact_check_id, weight=similarity_matrix_post_fact[i, j])

    return G

## Fact-Check Retrieval

In [21]:
def retrieve_fact_checks(graph, post_id, k=10):
    """
    Retrieve the top-k fact-check nodes connected to a given post in a graph.
    Args:
        graph (networkx.Graph): The graph containing posts and fact-check nodes.
        post_id (int or str): The ID of the post node to retrieve fact-checks for.
        k (int, optional): The number of top fact-checks to retrieve. Defaults to 10.
    Returns:
        list: A list of fact-check node IDs sorted by their edge weight to the post node.
    """

    neighbors = nx.neighbors(graph, post_id)
    fact_checks = [
        (neighbor, graph[post_id][neighbor]['weight'])
        for neighbor in neighbors
        if graph.nodes[neighbor]['node_type'] == 'fact_check'
    ]
    
    # Sort by weight and return top-k
    fact_checks = sorted(fact_checks, key=lambda x: x[1], reverse=True)[:k]
    return [fact_check_id for fact_check_id, _ in fact_checks]

def retrieve_for_all_posts(graph, posts_df, k=10):
    """
    Retrieve the top k fact checks for all posts in the given DataFrame.
    Args:
        graph: The graph structure used for retrieval.
        posts_df (pd.DataFrame): DataFrame containing post IDs.
        k (int, optional): The number of top fact checks to retrieve for each post. Defaults to 10.
    Returns:
        pd.DataFrame: DataFrame containing post IDs and their corresponding top k fact checks.
    """
    
    results = []

    for post_id in posts_df['post_id']:
        top_k_fact_checks = retrieve_fact_checks(graph, post_id, k)
        results.append({"post_id": post_id, "top_k_fact_checks": top_k_fact_checks})

    return pd.DataFrame(results)


## Precision Evaluation (Success-at-10)

In [22]:

def success_at_k(predictions_df, ground_truth_df, k=10):
    """
    Calculate the success rate at k for the given predictions.
    Args:
        predictions_df (pd.DataFrame): DataFrame containing the predictions with columns 'post_id' and 'top_k_fact_checks'.
        ground_truth_df (pd.DataFrame): DataFrame containing the ground truth with columns 'post_id' and 'fact_check_id'.
        k (int, optional): The number of top predictions to consider. Defaults to 10.
    Returns:
        float: The success rate at k, which is the proportion of posts for which at least one of the top k predicted fact checks is correct.
    """

    successes = 0

    for _, row in predictions_df.iterrows():
        post_id = row['post_id']
        predicted_fact_checks = set(row['top_k_fact_checks'])
        true_fact_checks = set(ground_truth_df[ground_truth_df['post_id'] == post_id]['fact_check_id'])
        
        if predicted_fact_checks.intersection(true_fact_checks):
            successes += 1

    return successes / len(predictions_df)

## Pipeline

### Load data

In [24]:

df_fact_checks, df_posts, df_fact_check_post_mapping = df_loader.load_data()

In [ ]:
fact_checks_mono_sample_1000=pd.read_csv('data/fact_checks_mono_sample_1000.csv')
posts_mono_sample_1000=pd.read_csv('data/posts_train_mono_sample_1000.csv')

embeddings_utils=Embeddings

embedding_columns_posts = ['ocr_text_embedding', 'ocr_text_eng_embedding']
embedding_columns_fact_checks = ['claim_title_eng_embedding', 'claim_title_embedding']

fact_checks_mono_sample_1000=embeddings_utils.restore_embeddings(fact_checks_mono_sample_1000, embedding_columns_fact_checks)
posts_mono_sample_1000=embeddings_utils.restore_embeddings(posts_mono_sample_1000, embedding_columns_posts)

In [26]:
# Write all the code for the pipeline here
graph = build_graph(posts_df=posts_mono_sample_1000, fact_checks_df=fact_checks_mono_sample_1000, post_embedding_col='ocr_text_embedding', fact_embedding_col='claim_title_embedding', threshold=0.7)
predictions = retrieve_for_all_posts(graph, posts_mono_sample_1000)

success_at_k(predictions, df_fact_check_post_mapping, k=10)

0.684

In [28]:
# Write all the code for the pipeline here
graph_85 = build_graph(posts_df=posts_mono_sample_1000, fact_checks_df=fact_checks_mono_sample_1000, post_embedding_col='ocr_text_embedding', fact_embedding_col='claim_title_embedding', threshold=0.85)
predictions_85 = retrieve_for_all_posts(graph_85, posts_mono_sample_1000)

success_at_k(predictions_85, df_fact_check_post_mapping, k=10)

0.273

In [29]:
# Write all the code for the pipeline here
graph_60 = build_graph(posts_df=posts_mono_sample_1000, fact_checks_df=fact_checks_mono_sample_1000, post_embedding_col='ocr_text_embedding', fact_embedding_col='claim_title_embedding', threshold=0.6)
predictions_60 = retrieve_for_all_posts(graph_60, posts_mono_sample_1000)

success_at_k(predictions_60, df_fact_check_post_mapping, k=10)

0.684

In [37]:
# Write all the code for the pipeline here
graph_faiss = build_graph(posts_df=posts_mono_sample_1000, fact_checks_df=fact_checks_mono_sample_1000, method='faiss',post_embedding_col='ocr_text_embedding', fact_embedding_col='claim_title_embedding', threshold=0.7)
predictions_faiss = retrieve_for_all_posts(graph_faiss, posts_mono_sample_1000)

success_at_k(predictions_faiss, df_fact_check_post_mapping, k=10)

0.0